!pip install requests beautifulsoup4 pandas

In [1]:
import os
import requests
import pandas as pd
import zipfile
from io import BytesIO

import threading

os.makedirs(os.path.join(os.getcwd() + "/data"), exist_ok=True)

In [2]:
def extractor_lic(
    year_ran=(2007, 2026), moth_ran=(1, 12), guardar_fisico=False, guardar_df=False
):

    if guardar_df:
        registro_df = dict()

    for year in range(year_ran[0], year_ran[1] + 1):
        for moth in range(moth_ran[0], moth_ran[1] + 1):

            url = f"https://transparenciachc.blob.core.windows.net/lic-da/{year}-{moth}.zip"

            try:
                respuesta = requests.get(url, timeout=120)
                respuesta.raise_for_status()

                with zipfile.ZipFile(BytesIO(respuesta.content)) as archivo_zip:

                    archivos_csv = [
                        nombre
                        for nombre in archivo_zip.namelist()
                        if nombre.lower().endswith(".csv")
                        and not nombre.endswith("/")
                    ]

                    print(f"{year}-{moth}: {len(archivos_csv)} CSV encontrados")


                    if guardar_fisico or guardar_df:
                        for nombre in archivos_csv:

                            if guardar_fisico:
                                nombre_destino = f"data/{nombre}"

                                with archivo_zip.open(nombre) as origen:
                                    with open(nombre_destino, "wb") as destino:
                                        destino.write(origen.read())


                            if guardar_df:
                                with archivo_zip.open(nombre) as archivo:
                                    df = pd.read_csv(
                                        archivo,
                                        sep=";",
                                        encoding="latin-1",
                                        quotechar='"',
                                        low_memory=False
                                    )
                                    registro_df[nombre] = df

                            print(f"  Guardado: {nombre}")

            except requests.HTTPError as e:
                print(
                    f"{year}-{moth}: HTTP "
                    f"{e.response.status_code}"
                )

            except (requests.RequestException, zipfile.BadZipFile) as e:
                print(f"{year}-{moth}: error: {e}")

    if guardar_df:
        return registro_df
            

In [3]:
hilos = []

for year in range(2007, 2026 + 1):
    hilo = threading.Thread(
        target=extractor_lic,
        kwargs={
            "year_ran": (year, year),
            "guardar_fisico": True
        }
    )

    hilos.append(hilo)
    hilo.start()

for hilo in hilos:
    hilo.join()

2025-1: 1 CSV encontrados
  Guardado: lic_2025-1.csv
2026-1: 1 CSV encontrados
  Guardado: lic_2026-1.csv
2025-2: 1 CSV encontrados
  Guardado: lic_2025-2.csv
2025-3: 1 CSV encontrados
  Guardado: lic_2025-3.csv
2021-1: 1 CSV encontrados
  Guardado: lic_2021-1.csv
2023-1: 1 CSV encontrados
2022-1: 1 CSV encontrados
  Guardado: lic_2023-1.csv  Guardado: lic_2022-1.csv

2021-2: 1 CSV encontrados
  Guardado: lic_2021-2.csv
2026-2: 1 CSV encontrados
  Guardado: lic_2026-2.csv
2024-1: 1 CSV encontrados
  Guardado: lic_2024-1.csv
2024-2: 1 CSV encontrados
  Guardado: lic_2024-2.csv
2019-1: 1 CSV encontrados
2018-1: 1 CSV encontrados
  Guardado: lic_2019-1.csv
  Guardado: lic_2018-1.csv
2024-3: 1 CSV encontrados
  Guardado: lic_2024-3.csv
2021-3: 1 CSV encontrados
  Guardado: lic_2021-3.csv
2020-1: 1 CSV encontrados
  Guardado: lic_2020-1.csv
2021-4: 1 CSV encontrados
  Guardado: lic_2021-4.csv
2019-2: 1 CSV encontrados
  Guardado: lic_2019-2.csv
2021-5: 1 CSV encontrados
  Guardado: lic_2021

In [7]:
df = pd.read_csv(
    "data/lic_2007-1.csv",
    sep=";",
    encoding="latin-1",
    quotechar='"',
    low_memory=False
)

df

,Codigo,Link,CodigoExterno,Nombre,Descripcion,Tipo de Adquisición,CodigoEstado,Estado,CodigoOrganismo,NombreOrganismo,...,Estado Oferta,Cantidad Ofertada,Moneda de la Oferta,MontoUnitarioOferta,Valor Total Ofertado,CantidadAdjudicada,MontoLineaAdjudica,FechaEnvioOferta,Estado final Oferta,Oferta seleccionada
0,1395950,http://www.mercadopublico.cl/fichaLicitacion.h...,713-157-CO06,Proyecto 45 Cod. Andina - Gob. Los Andes 2006,Comuna: San Esteban\r\r\nOrganización: Club De...,Licitación Pública Menor a 100 UTM (L1),8,Adjudicada,7009,Gobernación Provincial de Los Andes,...,Aceptada,4,Peso Chileno,18500,74000,0,0,2007-01-23,Perdedora,No Seleccionada
1,1395950,http://www.mercadopublico.cl/fichaLicitacion.h...,713-157-CO06,Proyecto 45 Cod. Andina - Gob. Los Andes 2006,Comuna: San Esteban\r\r\nOrganización: Club De...,Licitación Pública Menor a 100 UTM (L1),8,Adjudicada,7009,Gobernación Provincial de Los Andes,...,Aceptada,4,Peso Chileno,26307,105228,0,0,2007-01-23,Perdedora,No Seleccionada
2,1395950,http://www.mercadopublico.cl/fichaLicitacion.h...,713-157-CO06,Proyecto 45 Cod. Andina - Gob. Los Andes 2006,Comuna: San Esteban\r\r\nOrganización: Club De...,Licitación Pública Menor a 100 UTM (L1),8,Adjudicada,7009,Gobernación Provincial de Los Andes,...,Aceptada,4,Peso Chileno,18540,74160,0,0,2007-01-23,Perdedora,No Seleccionada
3,1395950,http://www.mercadopublico.cl/fichaLicitacion.h...,713-157-CO06,Proyecto 45 Cod. Andina - Gob. Los Andes 2006,Comuna: San Esteban\r\r\nOrganización: Club De...,Licitación Pública Menor a 100 UTM (L1),8,Adjudicada,7009,Gobernación Provincial de Los Andes,...,Aceptada,4,Peso Chileno,13365,53460,0,0,2007-01-23,Perdedora,No Seleccionada
4,1395950,http://www.mercadopublico.cl/fichaLicitacion.h...,713-157-CO06,Proyecto 45 Cod. Andina - Gob. Los Andes 2006,Comuna: San Esteban\r\r\nOrganización: Club De...,Licitación Pública Menor a 100 UTM (L1),8,Adjudicada,7009,Gobernación Provincial de Los Andes,...,Aceptada,4,Peso Chileno,13567,54268,4,54268,2007-01-23,Ganadora,Seleccionada
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827807,5238202,http://www.mercadopublico.cl/fichaLicitacion.h...,1381-13-R108,Solicitud 08/07 Matriales Ferretería año 2007,Según necesidad del Hospital de hacer un conve...,Licitación Pública Menor a 100 UTM (L1),7,Desierta (o art. 3 ó 9 Ley 19.886),7047,Hospital Gorbea,...,Aceptada,1,Peso Chileno,9411,9411,0,0,NaN,Perdedora,Perdedora
827808,5238202,http://www.mercadopublico.cl/fichaLicitacion.h...,1381-13-R108,Solicitud 08/07 Matriales Ferretería año 2007,Según necesidad del Hospital de hacer un conve...,Licitación Pública Menor a 100 UTM (L1),7,Desierta (o art. 3 ó 9 Ley 19.886),7047,Hospital Gorbea,...,Aceptada,1,Peso Chileno,1290,1290,0,0,NaN,Perdedora,Perdedora
827809,5238202,http://www.mercadopublico.cl/fichaLicitacion.h...,1381-13-R108,Solicitud 08/07 Matriales Ferretería año 2007,Según necesidad del Hospital de hacer un conve...,Licitación Pública Menor a 100 UTM (L1),7,Desierta (o art. 3 ó 9 Ley 19.886),7047,Hospital Gorbea,...,Aceptada,1,Peso Chileno,714,714,0,0,NaN,Perdedora,Perdedora
827810,5238202,http://www.mercadopublico.cl/fichaLicitacion.h...,1381-13-R108,Solicitud 08/07 Matriales Ferretería año 2007,Según necesidad del Hospital de hacer un conve...,Licitación Pública Menor a 100 UTM (L1),7,Desierta (o art. 3 ó 9 Ley 19.886),7047,Hospital Gorbea,...,Aceptada,1,Peso Chileno,3590,3590,0,0,NaN,Perdedora,Perdedora


In [8]:
print(df.columns.to_list())

['Codigo', 'Link', 'CodigoExterno', 'Nombre', 'Descripcion', 'Tipo de Adquisición', 'CodigoEstado', 'Estado', 'CodigoOrganismo', 'NombreOrganismo', 'RutUnidad', 'CodigoUnidad', 'NombreUnidad', 'DireccionUnidad', 'ComunaUnidad', 'RegionUnidad', 'Informada', 'CodigoTipo', 'Tipo', 'TipoConvocatoria', 'CodigoMoneda', 'Moneda Adquisición', 'Etapas', 'EstadoEtapas', 'TomaRazon', 'EstadoPublicidadOfertas', 'JustificacionPublicidad', 'EstadoCS', 'Contrato', 'Obras', 'CantidadReclamos', 'FechaCreacion', 'FechaCierre', 'FechaInicio', 'FechaFinal', 'FechaPubRespuestas', 'FechaActoAperturaTecnica', 'FechaActoAperturaEconomica', 'FechaPublicacion', 'FechaAdjudicacion', 'FechaEstimadaAdjudicacion', 'FechaSoporteFisico', 'FechaTiempoEvaluacion', 'UnidadTiempoEvaluacion', 'FechaEstimadaFirma', 'FechasUsuario', 'FechaVisitaTerreno', 'DireccionVisita', 'FechaEntregaAntecedentes', 'DireccionEntrega', 'Estimacion', 'FuenteFinanciamiento', 'VisibilidadMonto', 'MontoEstimado', 'Tiempo', 'UnidadTiempo', 'Mod